In [ ]:
#INSTALLS
!pip -q install python-louvain imageio

# IMPORTS
from google.colab import drive
drive.mount('/content/drive')
import os, ast, pickle, imageio, time
import pandas as pd
import numpy as np
import networkx as nx
from itertools import combinations
from collections import Counter
import matplotlib.pyplot as plt
from scipy.stats import linregress
from networkx.algorithms.community import louvain_communities

FILE_PATH   = "/content/bananika.csv"
OUTPUT_BASE = "/content/drive/MyDrive/temporal_star_results4"
os.makedirs(OUTPUT_BASE, exist_ok=True)

TOP_K = 15
WINDOW = 1
RANDOM_SEED = 0

# LOAD DATA
df = pd.read_csv(FILE_PATH, engine="python", encoding="latin-1",on_bad_lines="skip")
df = df.loc[:, ~df.columns.str.match(r"^Unnamed")].copy()
df["year"] = pd.to_numeric(df["year"], errors="coerce")
df = df.dropna(subset=["year"])
df["year"] = df["year"].astype(int)

# AUTHOR PARSER
def parse_authors(cell):
    if pd.isna(cell): return []
    s = str(cell).strip()
    if s.startswith("[") and s.endswith("]"):
        try:
            return [a.strip() for a in ast.literal_eval(s)]
        except:
            return []
    s = s.replace(" and ", ", ").replace(";", ",")
    return list({a.strip() for a in s.split(",") if a.strip()})

# BUILD YEARLY GRAPHS
def build_graph(frame):
    edge_w = Counter()
    for _, row in frame.iterrows():
        authors = parse_authors(row["authors"])
        if len(authors) < 2: continue
        for u,v in combinations(sorted(set(authors)),2):
            edge_w[(u,v)] += 1.0
    G = nx.Graph()
    for (u,v), w in edge_w.items():
        G.add_edge(u,v,weight=w)
    return G

years = sorted(df["year"].unique())
year_graphs = {}
for y in years:
    G = build_graph(df[df["year"]==y])
    year_graphs[y] = G

print("Built yearly graphs:", len(year_graphs))

#CENTRALITIES
def compute_metrics(G):
    if G.number_of_nodes()==0:
        return {}
    deg = nx.degree_centrality(G)
    btw = nx.betweenness_centrality(G, normalized=True)
    clo = nx.closeness_centrality(G)
    try:
        eig = nx.eigenvector_centrality(G, max_iter=1000)
    except:
        eig = nx.pagerank(G)
    return {
        "degree":deg,
        "betweenness":btw,
        "closeness":clo,
        "eigenvector":eig
    }

records=[]
for y,G in year_graphs.items():
    mets = compute_metrics(G)
    for metric,vals in mets.items():
        for a,v in vals.items():
            records.append({
                "author":a,
                "year":y,
                "metric":metric,
                "value":v
            })

metrics_df = pd.DataFrame(records)
metrics_df.to_csv(os.path.join(OUTPUT_BASE,"all_yearly_metrics.csv"),index=False)

#STAR ANALYSIS
def star_analysis(df_metric):
    rows=[]
    for author,g in df_metric.groupby("author"):
        if len(g)<5: continue
        g=g.sort_values("year")
        slope,_,r,p,_ = linregress(g["year"],g["value"])
        rows.append({
            "author":author,
            "slope":slope,
            "r":r,
            "p":p,
            "years":len(g)
        })
    return pd.DataFrame(rows).sort_values("slope",ascending=False)

all_star_results={}

for metric in ["degree","betweenness","closeness","eigenvector"]:
    dfm = metrics_df[metrics_df.metric==metric]
    slopes = star_analysis(dfm)
    rising = slopes.head(TOP_K)
    dying  = slopes.tail(TOP_K).sort_values("slope")
    rising.to_csv(os.path.join(OUTPUT_BASE,f"{metric}_top_rising.csv"),index=False)
    dying.to_csv(os.path.join(OUTPUT_BASE,f"{metric}_top_dying.csv"),index=False)
    all_star_results[metric]=(rising,dying)

print("Saved rising/dying star tables.")

#NETWORK GIF PER YEAR
def make_yearly_gif():
    frames=[]
    for y,G in year_graphs.items():
        if G.number_of_nodes()==0: continue

        pos = nx.spring_layout(G, seed=RANDOM_SEED)

        communities = louvain_communities(G, seed=RANDOM_SEED)

        comm_map = {}
        for i, c in enumerate(communities):
            for n in c:
                comm_map[n] = i

        node_colors = [comm_map.get(n, 0) for n in G.nodes()]

        plt.figure(figsize=(8,8))
        nx.draw_networkx_edges(G, pos, alpha=0.15)
        nx.draw_networkx_nodes(G, pos,
                               node_color=node_colors,
                               cmap="tab20",
                               node_size=30)

        plt.title(f"Collaboration Network {y}")
        plt.axis("off")
        plt.tight_layout()
        plt.savefig(f"tmp_{y}.png", dpi=200)
        frames.append(imageio.imread(f"tmp_{y}.png"))
        plt.close()

    imageio.mimsave(os.path.join(OUTPUT_BASE,"yearly_collaboration.gif"),
                    frames,
                    fps=0.5)

make_yearly_gif()

print("GIF saved.")
print("ALL RESULTS SAVED TO:",OUTPUT_BASE)

Mounted at /content/drive
Built yearly graphs: 38
Saved rising/dying star tables.


/tmp/ipython-input-363450383.py:156: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  frames.append(imageio.imread(f"tmp_{y}.png"))


GIF saved.
ALL RESULTS SAVED TO: /content/drive/MyDrive/temporal_star_results4


In [ ]:
df['year'].unique()

array([1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997,
       1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008,
       2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019,
       2020, 2021, 2022, 2023])